# BDC Statistics Explore 2026 — SigLIP2 Source-Robust Severity V2 (Single GPU)

Final-train notebook for one GPU. The recipe is fixed from the previous TRAIN-only analysis and grouped-CV experiment:

- `google/siglip2-base-patch16-384`
- aspect-ratio-preserving 384px input
- shared disaster representation
- **global severity head + bounded disaster-specific residual heads**
- duplicate/component-aware training weights
- conflict-aware severity weighting
- mild severity label smoothing
- targeted JPEG/source augmentation
- random train-time letterbox placement
- frozen-head warmup → partial fine-tuning of the last 4 vision blocks
- probability blend of partial epochs 3 and 4

The notebook trains on the full labelled TRAIN directory. TEST is treated only as an unlabeled inference set. No filename-range rules, TEST-derived calibration, pseudo-labels, or TRAIN↔TEST lookup logic are used.


## Dataset paths

Expected layout:

- `/workspace/dataset/SE/TRAIN/<DISASTER>/<SEVERITY>/*`
- `/workspace/dataset/SE/TEST/*`
- `/workspace/dataset/SE/TRAIN/Solution.csv`

Submission encoding is explicit and never inferred from TEST predictions.


In [1]:
!nvidia-smi


Thu Sep  3 04:46:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   35C    P8              8W /  500W |       2MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Keep the Vast.ai PyTorch/CUDA build intact. Only install lightweight Python dependencies.
%pip install -q -U "transformers==4.57.1" "huggingface_hub>=0.34.0" pandas pillow imagehash tqdm safetensors


Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import os, sys, json, math, random, time, hashlib, io
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import imagehash
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import SiglipVisionModel, AutoImageProcessor
from huggingface_hub import snapshot_download

TRAIN_DIR = Path('/workspace/dataset/SE/TRAIN')
TEST_DIR = Path('/workspace/dataset/SE/TEST')
SOLUTION_PATH = Path('/workspace/dataset/SE/TRAIN/Solution.csv')
OUTPUT_DIR = Path('/workspace/output/siglip2_b384_source_robust_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DIR.is_dir(), f'Missing TRAIN_DIR: {TRAIN_DIR}'
assert TEST_DIR.is_dir(), f'Missing TEST_DIR: {TEST_DIR}'
assert SOLUTION_PATH.is_file(), f'Missing SOLUTION_PATH: {SOLUTION_PATH}'

print('TRAIN   :', TRAIN_DIR)
print('TEST    :', TEST_DIR)
print('SOLUTION:', SOLUTION_PATH)
print('OUTPUT  :', OUTPUT_DIR)


TRAIN   : /workspace/dataset/SE/TRAIN
TEST    : /workspace/dataset/SE/TEST
SOLUTION: /workspace/dataset/SE/TRAIN/Solution.csv
OUTPUT  : /workspace/output/siglip2_b384_source_robust_v2


## Single-GPU runtime policy

The statistical recipe keeps an effective batch of 32. Micro-batch is adjusted only for VRAM portability; gradient accumulation preserves the effective batch. BF16 is used when the GPU supports it, otherwise FP16 is used.


In [4]:
assert torch.cuda.is_available(), 'A CUDA GPU is required.'
assert torch.cuda.device_count() >= 1

torch.cuda.set_device(0)
DEVICE = torch.device('cuda:0')
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / (1024**3)

if VRAM_GB >= 40:
    MICRO_BATCH = 32
elif VRAM_GB >= 20:
    MICRO_BATCH = 16
elif VRAM_GB >= 12:
    MICRO_BATCH = 8
else:
    MICRO_BATCH = 4

EFFECTIVE_BATCH = 32
assert EFFECTIVE_BATCH % MICRO_BATCH == 0
ACCUM_STEPS = EFFECTIVE_BATCH // MICRO_BATCH
EVAL_BATCH = min(64, max(16, MICRO_BATCH * 2))
NUM_WORKERS = min(12, max(4, (os.cpu_count() or 8) // 4))
USE_GRAD_CHECKPOINTING = VRAM_GB < 20
USE_BF16 = bool(torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

# Throughput-friendly settings. They do not change model architecture.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('gpu:', props.name)
print(f'VRAM: {VRAM_GB:.1f} GiB')
print('AMP:', 'BF16' if USE_BF16 else 'FP16')
print('micro batch:', MICRO_BATCH)
print('gradient accumulation:', ACCUM_STEPS)
print('effective batch:', EFFECTIVE_BATCH)
print('eval batch:', EVAL_BATCH)
print('workers:', NUM_WORKERS)
print('gradient checkpointing:', USE_GRAD_CHECKPOINTING)


torch: 2.11.0+cu128
cuda build: 12.8
gpu: NVIDIA GeForce RTX 5090
VRAM: 31.4 GiB
AMP: BF16
micro batch: 16
gradient accumulation: 2
effective batch: 32
eval batch: 32
workers: 4
gradient checkpointing: False


## Model checkpoint prefetch

The FixRes 384 checkpoint is loaded with `SiglipVisionModel`. This is intentional: the checkpoint exposes the SigLIP-compatible vision configuration used by this model family.


In [5]:
MODEL_ID = 'google/siglip2-base-patch16-384'
cache_path = snapshot_download(MODEL_ID)
print('Cached model at:', cache_path)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

Cached model at: /root/.cache/huggingface/hub/models--google--siglip2-base-patch16-384/snapshots/f775b65a79762255128c981547af89addcfe0f88


## Build a TRAIN-only duplicate/component manifest

Exact and conservative perceptual components are used **only inside TRAIN** to prevent repeated collections from dominating the loss. No image is deleted or relabelled.

Weighting policy:

- base weight: `1 / sqrt(component_size)`
- normalized within each of the 9 joint classes to preserve class-scale balance
- severity-conflict component: severity contribution × `0.25`
- disaster-conflict component: disaster contribution × `0.25`


In [6]:
SEED = 20260901
IMAGE_EXTS = {'.jpg','.jpeg','.png','.jfif','.bmp','.webp'}
JENIS = ['BANJIR','GEMPA BUMI','KEBAKARAN']
KERUSAKAN = ['KERUSAKAN RINGAN','KERUSAKAN SEDANG','KERUSAKAN BERAT']
JENIS_TO_IDX = {x:i for i,x in enumerate(JENIS)}
KER_TO_IDX = {x:i for i,x in enumerate(KERUSAKAN)}
IDX_TO_JENIS = {i:x for i,x in enumerate(JENIS)}
IDX_TO_KER = {i:x for i,x in enumerate(KERUSAKAN)}

# Explicit competition-output encoding.
SUB_JENIS = {'BANJIR':1, 'GEMPA BUMI':2, 'KEBAKARAN':3}
SUB_KER = {'KERUSAKAN BERAT':1, 'KERUSAKAN RINGAN':2, 'KERUSAKAN SEDANG':3}

HASH_CACHE = OUTPUT_DIR / 'train_hashes.csv'
MANIFEST_PATH = OUTPUT_DIR / 'train_weighted_manifest.csv'


def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def enumerate_train(root):
    rows=[]
    for jenis in JENIS:
        for ker in KERUSAKAN:
            d=root/jenis/ker
            assert d.is_dir(), f'Missing class directory: {d}'
            for p in sorted(d.iterdir()):
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    ji=JENIS_TO_IDX[jenis]; ki=KER_TO_IDX[ker]
                    rows.append({
                        'path':str(p), 'jenis':jenis, 'kerusakan':ker,
                        'jenis_idx':ji, 'kerusakan_idx':ki,
                        'joint_idx':ji*3+ki,
                    })
    return pd.DataFrame(rows)

train_df = enumerate_train(TRAIN_DIR)
print('TRAIN images:', len(train_df))
display(train_df.groupby(['jenis','kerusakan']).size().unstack(fill_value=0))


def robust_rgb(im):
    im = ImageOps.exif_transpose(im)
    has_alpha = im.mode in ('RGBA','LA') or (im.mode == 'P' and 'transparency' in im.info)
    if has_alpha:
        rgba = im.convert('RGBA')
        bg = Image.new('RGBA', rgba.size, (128,128,128,255))
        return Image.alpha_composite(bg, rgba).convert('RGB')
    return im.convert('RGB')


def hash_one(path):
    p=Path(path)
    sha=hashlib.sha256(p.read_bytes()).hexdigest()
    with Image.open(p) as im:
        im=robust_rgb(im)
        return sha, str(imagehash.phash(im)), str(imagehash.dhash(im)), str(imagehash.average_hash(im))

if HASH_CACHE.exists():
    hdf=pd.read_csv(HASH_CACHE, dtype=str)
    print('Loaded hash cache:', HASH_CACHE)
else:
    workers=min(32, os.cpu_count() or 8)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        vals=list(tqdm(ex.map(hash_one, train_df.path), total=len(train_df), desc='Hashing TRAIN'))
    hdf=pd.DataFrame(vals, columns=['sha256','phash','dhash','ahash'])
    hdf.insert(0,'path',train_df.path.values)
    hdf.to_csv(HASH_CACHE,index=False)
    print('Saved hash cache:', HASH_CACHE)

assert list(hdf.path) == list(train_df.path), 'Hash cache path order mismatch.'

class DSU:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.r[a]<self.r[b]:a,b=b,a
        self.p[b]=a
        if self.r[a]==self.r[b]:self.r[a]+=1

def ham(a,b): return (int(a)^int(b)).bit_count()

def build_groups(hdf):
    n=len(hdf); dsu=DSU(n)
    sha_map={}
    for i,s in enumerate(hdf.sha256):
        if s in sha_map: dsu.union(i,sha_map[s])
        else: sha_map[s]=i

    ph=np.array([int(x,16) for x in hdf.phash], dtype=object)
    dh=np.array([int(x,16) for x in hdf.dhash], dtype=object)
    ah=np.array([int(x,16) for x in hdf.ahash], dtype=object)

    buckets={}; candidates=set()
    for i,v0 in enumerate(ph):
        v=int(v0)
        for band in range(4):
            key=(band,(v>>(16*band))&0xFFFF)
            for j in buckets.get(key,[]): candidates.add((j,i))
            buckets.setdefault(key,[]).append(i)

    kept=0
    for a,b in tqdm(candidates, desc='Strict near-duplicate edges'):
        if ham(ph[a],ph[b])<=1 and ham(dh[a],dh[b])<=2 and ham(ah[a],ah[b])<=4:
            dsu.union(a,b); kept+=1

    roots=[dsu.find(i) for i in range(n)]
    remap={r:j for j,r in enumerate(sorted(set(roots)))}
    groups=np.array([remap[r] for r in roots], dtype=np.int64)
    return groups, kept

if MANIFEST_PATH.exists():
    manifest=pd.read_csv(MANIFEST_PATH)
    print('Loaded weighted manifest:', MANIFEST_PATH)
else:
    scene_group, near_edges = build_groups(hdf)
    manifest=train_df.copy()
    manifest['scene_group']=scene_group

    gsize=manifest.groupby('scene_group').size().rename('group_size')
    manifest=manifest.join(gsize,on='scene_group')

    sev_n=manifest.groupby('scene_group').kerusakan_idx.nunique().rename('severity_nlabels')
    dis_n=manifest.groupby('scene_group').jenis_idx.nunique().rename('disaster_nlabels')
    manifest=manifest.join(sev_n,on='scene_group').join(dis_n,on='scene_group')
    manifest['severity_conflict']=manifest.severity_nlabels.gt(1)
    manifest['disaster_conflict']=manifest.disaster_nlabels.gt(1)

    manifest['component_weight_raw']=1.0/np.sqrt(manifest.group_size.astype(float))
    # Normalize within each joint class so component balancing does not silently alter class scale.
    denom=manifest.groupby('joint_idx').component_weight_raw.transform('mean')
    manifest['component_weight']=manifest.component_weight_raw/denom
    manifest['component_weight']=manifest.component_weight.clip(0.20,2.50)
    # Re-normalize after clipping.
    manifest['component_weight'] /= manifest.groupby('joint_idx').component_weight.transform('mean')

    manifest['severity_weight']=manifest.component_weight*np.where(manifest.severity_conflict,0.25,1.0)
    manifest['disaster_weight']=manifest.component_weight*np.where(manifest.disaster_conflict,0.25,1.0)

    manifest.to_csv(MANIFEST_PATH,index=False)
    print('strict near edges:',near_edges)
    print('Saved weighted manifest:',MANIFEST_PATH)

assert len(manifest)==len(train_df)
assert manifest.path.is_unique
print('scene groups:',manifest.scene_group.nunique())
print('images in non-singleton groups:',int((manifest.group_size>1).sum()))
print('severity-conflict images:',int(manifest.severity_conflict.sum()))
print('disaster-conflict images:',int(manifest.disaster_conflict.sum()))
print('\nWeight summary:')
display(manifest[['component_weight','severity_weight','disaster_weight']].describe())


TRAIN images: 17482


kerusakan,KERUSAKAN BERAT,KERUSAKAN RINGAN,KERUSAKAN SEDANG
jenis,,,
BANJIR,1968,2018,1971
GEMPA BUMI,1623,1393,2730
KEBAKARAN,2025,1746,2008


Hashing TRAIN:   0%|          | 0/17482 [00:00<?, ?it/s]

Saved hash cache: /workspace/output/siglip2_b384_source_robust_v2/train_hashes.csv


Strict near-duplicate edges:   0%|          | 0/152663 [00:00<?, ?it/s]

strict near edges: 4111
Saved weighted manifest: /workspace/output/siglip2_b384_source_robust_v2/train_weighted_manifest.csv
scene groups: 15296
images in non-singleton groups: 3701
severity-conflict images: 376
disaster-conflict images: 2

Weight summary:


,component_weight,severity_weight,disaster_weight
count,17482.000000,17482.000000,17482.000000
mean,1.000000,0.990084,0.999934
std,0.171668,0.202364,0.171867
min,0.199972,0.064175,0.182338
25%,1.009825,1.009825,1.009825
50%,1.018508,1.018508,1.018508
75%,1.035004,1.035004,1.035004
max,1.330442,1.330442,1.330442


## Model, augmentation, and loss

The severity branch has one shared global classifier plus three bounded residual corrections. Specialists therefore adjust a shared damage representation instead of learning completely independent severity classifiers.

Training loss:

`0.30 × disaster CE + 0.55 × conditional severity CE + 0.15 × global severity CE`

Severity CE uses label smoothing `0.05`. Component/conflict weights are applied per image.


In [7]:
MODEL_ID='google/siglip2-base-patch16-384'
IMAGE_SIZE=384
PAD_RGB=(128,128,128)
DROPOUT=0.15
UNFREEZE_LAST_N=4

HEAD_EPOCHS=2
PARTIAL_EPOCHS=4
HEAD_LR=7e-4
PARTIAL_HEAD_LR=1e-4
PARTIAL_BACKBONE_LR=1e-5
WEIGHT_DECAY=0.05
WARMUP_RATIO=0.08
GRAD_CLIP=1.0

W_DISASTER=0.30
W_COND_SEVERITY=0.55
W_GLOBAL_SEVERITY=0.15
SEVERITY_LABEL_SMOOTHING=0.05
RESIDUAL_ALPHA=0.50

HFLIP_P=0.50
BRIGHTNESS=0.10
CONTRAST=0.10
SATURATION=0.07
JPEG_P=0.35
JPEG_Q_MIN=70
JPEG_Q_MAX=95

processor=AutoImageProcessor.from_pretrained(MODEL_ID,local_files_only=True)
mean=np.array(processor.image_mean,dtype=np.float32).reshape(3,1,1)
std=np.array(processor.image_std,dtype=np.float32).reshape(3,1,1)


def jpeg_perturb(im):
    q=random.randint(JPEG_Q_MIN,JPEG_Q_MAX)
    buf=io.BytesIO()
    im.save(buf,format='JPEG',quality=q,subsampling=2)
    buf.seek(0)
    with Image.open(buf) as rec:
        return rec.convert('RGB').copy()


def decode(path,train=False):
    with Image.open(path) as src:
        im=robust_rgb(src)

    if train:
        if random.random()<HFLIP_P:
            im=ImageOps.mirror(im)
        if random.random()<JPEG_P:
            # Re-encode regardless of original suffix to weaken file/source-style shortcuts.
            im=jpeg_perturb(im)
        im=ImageEnhance.Brightness(im).enhance(1+random.uniform(-BRIGHTNESS,BRIGHTNESS))
        im=ImageEnhance.Contrast(im).enhance(1+random.uniform(-CONTRAST,CONTRAST))
        im=ImageEnhance.Color(im).enhance(1+random.uniform(-SATURATION,SATURATION))

    w,h=im.size
    scale=min(IMAGE_SIZE/w,IMAGE_SIZE/h)
    nw=max(1,round(w*scale)); nh=max(1,round(h*scale))
    im=im.resize((nw,nh),Image.Resampling.BICUBIC)

    free_x=IMAGE_SIZE-nw; free_y=IMAGE_SIZE-nh
    if train:
        ox=random.randint(0,free_x) if free_x>0 else 0
        oy=random.randint(0,free_y) if free_y>0 else 0
    else:
        ox=free_x//2; oy=free_y//2

    canvas=Image.new('RGB',(IMAGE_SIZE,IMAGE_SIZE),PAD_RGB)
    canvas.paste(im,(ox,oy))
    arr=np.asarray(canvas,dtype=np.float32).transpose(2,0,1)/255.0
    arr=(arr-mean)/std
    return torch.from_numpy(arr)


class TrainDS(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        return {
            'pixel_values':decode(r.path,train=True),
            'jenis_idx':int(r.jenis_idx),
            'kerusakan_idx':int(r.kerusakan_idx),
            'disaster_weight':float(r.disaster_weight),
            'severity_weight':float(r.severity_weight),
        }


class TestDS(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        return {'pixel_values':decode(r.path,train=False),'row_index':i}


class SourceRobustModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision=SiglipVisionModel.from_pretrained(MODEL_ID,local_files_only=True)
        if USE_GRAD_CHECKPOINTING and hasattr(self.vision,'gradient_checkpointing_enable'):
            self.vision.gradient_checkpointing_enable()
        d=self.vision.config.hidden_size
        self.feature_norm=nn.LayerNorm(d)
        self.drop=nn.Dropout(DROPOUT)
        self.disaster=nn.Linear(d,3)
        self.global_severity=nn.Linear(d,3)
        self.severity_residual=nn.ModuleList([nn.Linear(d,3) for _ in range(3)])

    def features(self,pixel_values):
        out=self.vision(pixel_values=pixel_values,return_dict=True)
        z=out.pooler_output if getattr(out,'pooler_output',None) is not None else out.last_hidden_state.mean(1)
        return self.drop(self.feature_norm(z.float()))

    def forward(self,pixel_values):
        z=self.features(pixel_values)
        disaster_logits=self.disaster(z)
        global_logits=self.global_severity(z)
        residual=torch.stack([torch.tanh(h(z)) for h in self.severity_residual],dim=1) # B,3,3
        cond_logits=global_logits.unsqueeze(1)+RESIDUAL_ALPHA*residual
        disaster_prob=F.softmax(disaster_logits,dim=-1)
        severity_prob=torch.sum(disaster_prob.unsqueeze(-1)*F.softmax(cond_logits,dim=-1),dim=1)
        return disaster_logits,global_logits,cond_logits,severity_prob


def set_stage(model,stage):
    for p in model.vision.parameters(): p.requires_grad=False
    for module in [model.feature_norm,model.disaster,model.global_severity,model.severity_residual]:
        for p in module.parameters(): p.requires_grad=True

    if stage=='partial':
        core=getattr(model.vision,'vision_model',model.vision)
        if hasattr(core,'encoder') and hasattr(core.encoder,'layers'):
            layers=core.encoder.layers
        elif hasattr(model.vision,'encoder') and hasattr(model.vision.encoder,'layers'):
            layers=model.vision.encoder.layers
        else:
            raise AttributeError('Could not locate SigLIP vision transformer layers.')
        for layer in layers[-UNFREEZE_LAST_N:]:
            for p in layer.parameters(): p.requires_grad=True
        for attr in ('post_layernorm','head'):
            module=getattr(core,attr,None)
            if module is not None:
                for p in module.parameters(): p.requires_grad=True


def weighted_mean(x,w):
    return (x*w).sum()/w.sum().clamp_min(1e-6)


def loss_fn(disaster_logits,global_logits,cond_logits,yj,yk,wd,wk):
    lj=F.cross_entropy(disaster_logits,yj,reduction='none')
    global_sev=F.cross_entropy(global_logits,yk,reduction='none',label_smoothing=SEVERITY_LABEL_SMOOTHING)
    cond_true=cond_logits[torch.arange(len(yj),device=yj.device),yj]
    cond_sev=F.cross_entropy(cond_true,yk,reduction='none',label_smoothing=SEVERITY_LABEL_SMOOTHING)
    return (
        W_DISASTER*weighted_mean(lj,wd)
        + W_COND_SEVERITY*weighted_mean(cond_sev,wk)
        + W_GLOBAL_SEVERITY*weighted_mean(global_sev,wk)
    )


def make_optimizer(model,stage):
    head=[]; backbone=[]
    for name,p in model.named_parameters():
        if not p.requires_grad: continue
        (backbone if name.startswith('vision.') else head).append(p)
    groups=[]
    if backbone:
        groups.append({'params':backbone,'lr':PARTIAL_BACKBONE_LR if stage=='partial' else HEAD_LR})
    if head:
        groups.append({'params':head,'lr':PARTIAL_HEAD_LR if stage=='partial' else HEAD_LR})
    return torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)


def cosine_scheduler(opt,total_updates):
    warm=max(1,int(total_updates*WARMUP_RATIO))
    def fn(step):
        if step<warm: return max(1e-8,step/warm)
        prog=(step-warm)/max(1,total_updates-warm)
        return 0.5*(1+math.cos(math.pi*min(1.0,prog)))
    return torch.optim.lr_scheduler.LambdaLR(opt,fn)


def train_one_epoch(model,loader,opt,sched,scaler):
    model.train(); opt.zero_grad(set_to_none=True)
    total_loss=0.0; n=0; updates=0
    pbar=tqdm(enumerate(loader),total=len(loader),leave=False)
    for step,b in pbar:
        x=b['pixel_values'].to(DEVICE,non_blocking=True)
        yj=b['jenis_idx'].to(DEVICE,non_blocking=True)
        yk=b['kerusakan_idx'].to(DEVICE,non_blocking=True)
        wd=b['disaster_weight'].to(DEVICE,dtype=torch.float32,non_blocking=True)
        wk=b['severity_weight'].to(DEVICE,dtype=torch.float32,non_blocking=True)

        with torch.autocast('cuda',dtype=AMP_DTYPE):
            j,g,c,_=model(x)
            loss=loss_fn(j,g,c,yj,yk,wd,wk)
            backward_loss=loss/ACCUM_STEPS

        if scaler.is_enabled(): scaler.scale(backward_loss).backward()
        else: backward_loss.backward()

        should_step=((step+1)%ACCUM_STEPS==0) or (step+1==len(loader))
        if should_step:
            if scaler.is_enabled():
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                opt.step()
            opt.zero_grad(set_to_none=True)
            sched.step(); updates+=1

        total_loss += float(loss.detach())*len(x); n += len(x)
        pbar.set_postfix(loss=f'{total_loss/max(1,n):.4f}',updates=updates)
    return total_loss/max(1,n)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Loader smoke test

This catches model/config incompatibilities before a paid training run begins.


In [8]:
seed_all()
model=SourceRobustModel().to(DEVICE)
model.eval()
x=torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE)
with torch.no_grad(),torch.autocast('cuda',dtype=AMP_DTYPE):
    j,g,c,k=model(x)
assert j.shape==(1,3) and g.shape==(1,3) and c.shape==(1,3,3) and k.shape==(1,3)
core=getattr(model.vision,'vision_model',model.vision)
layers=getattr(getattr(core,'encoder',None),'layers',None)
assert layers is not None and len(layers)>=UNFREEZE_LAST_N
print('SMOKE OK')
print('feature dim:',model.disaster.in_features)
print('vision blocks:',len(layers))
del model,x,j,g,c,k
torch.cuda.empty_cache()


SMOKE OK
feature dim: 768
vision blocks: 12


## Full-TRAIN training

This run uses all labelled TRAIN images. Epoch counts are fixed from the prior TRAIN-only grouped-CV experiment:

- 2 frozen-head epochs
- 4 partial-fine-tuning epochs
- save partial epoch 3 and partial epoch 4 for the final snapshot probability blend

If both snapshot checkpoints already exist, training is skipped to avoid accidentally paying twice.


In [9]:
seed_all()
SNAP3=OUTPUT_DIR/'partial_epoch_3.pt'
SNAP4=OUTPUT_DIR/'partial_epoch_4.pt'
HISTORY_PATH=OUTPUT_DIR/'train_history.json'

if SNAP3.exists() and SNAP4.exists():
    print('Both final snapshots already exist. Skipping training.')
    print(SNAP3)
    print(SNAP4)
else:
    train_ds=TrainDS(manifest)
    gen=torch.Generator(); gen.manual_seed(SEED)
    train_loader=DataLoader(
        train_ds,batch_size=MICRO_BATCH,shuffle=True,num_workers=NUM_WORKERS,
        pin_memory=True,persistent_workers=(NUM_WORKERS>0),drop_last=False,generator=gen,
    )

    model=SourceRobustModel().to(DEVICE)
    scaler=torch.amp.GradScaler('cuda',enabled=not USE_BF16)
    history=[]

    # Stage 1: train only heads / task adapters.
    set_stage(model,'heads')
    opt=make_optimizer(model,'heads')
    updates_per_epoch=math.ceil(len(train_loader)/ACCUM_STEPS)
    sched=cosine_scheduler(opt,updates_per_epoch*HEAD_EPOCHS)
    for ep in range(1,HEAD_EPOCHS+1):
        t=time.time()
        loss=train_one_epoch(model,train_loader,opt,sched,scaler)
        rec={'stage':'heads','epoch':ep,'loss':loss,'minutes':(time.time()-t)/60}
        history.append(rec); print(rec)

    # Stage 2: last four SigLIP vision blocks + heads.
    set_stage(model,'partial')
    opt=make_optimizer(model,'partial')
    sched=cosine_scheduler(opt,updates_per_epoch*PARTIAL_EPOCHS)
    for ep in range(1,PARTIAL_EPOCHS+1):
        t=time.time()
        loss=train_one_epoch(model,train_loader,opt,sched,scaler)
        rec={'stage':'partial','epoch':ep,'loss':loss,'minutes':(time.time()-t)/60}
        history.append(rec); print(rec)

        # Save lightweightly useful snapshots for the fixed epoch-3/4 probability ensemble.
        if ep in (3,4):
            path=SNAP3 if ep==3 else SNAP4
            torch.save({
                'model':model.state_dict(),
                'stage':'partial','epoch':ep,
                'config':{
                    'model_id':MODEL_ID,'image_size':IMAGE_SIZE,'residual_alpha':RESIDUAL_ALPHA,
                    'severity_label_smoothing':SEVERITY_LABEL_SMOOTHING,
                    'effective_batch':EFFECTIVE_BATCH,
                }
            },path)
            print('Saved:',path)

    HISTORY_PATH.write_text(json.dumps(history,indent=2))
    print('Saved history:',HISTORY_PATH)
    del model,train_loader,train_ds,opt,sched,scaler
    torch.cuda.empty_cache()

assert SNAP3.exists() and SNAP4.exists(), 'Expected epoch-3 and epoch-4 snapshots.'


  0%|          | 0/1093 [00:00<?, ?it/s]

{'stage': 'heads', 'epoch': 1, 'loss': 0.5220182162784436, 'minutes': 1.8481937130292256}


  0%|          | 0/1093 [00:00<?, ?it/s]

{'stage': 'heads', 'epoch': 2, 'loss': 0.42415631998800435, 'minutes': 1.826312287648519}


  0%|          | 0/1093 [00:00<?, ?it/s]

...

{'stage': 'partial', 'epoch': 2, 'loss': 0.310658566494335, 'minutes': 1.866451863447825}


  0%|          | 0/1093 [00:00<?, ?it/s]

{'stage': 'partial', 'epoch': 3, 'loss': 0.25676198643113884, 'minutes': 1.9090462764104208}
Saved: /workspace/output/siglip2_b384_source_robust_v2/partial_epoch_3.pt


  0%|          | 0/1093 [00:00<?, ?it/s]

{'stage': 'partial', 'epoch': 4, 'loss': 0.22665509867280834, 'minutes': 1.884256092707316}
Saved: /workspace/output/siglip2_b384_source_robust_v2/partial_epoch_4.pt
Saved history: /workspace/output/siglip2_b384_source_robust_v2/train_history.json


## TEST inference: snapshot probability blend

Inference is deterministic, centered letterbox, no flip TTA. The two selected full-TRAIN snapshots are averaged at the probability level.


In [11]:
def enumerate_test(root):
    rows = []
    for p in root.iterdir():
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            rows.append({
                'path': str(p),
                'id': str(p.stem),
            })

    def sort_key(r):
        return (0, int(r['id'])) if r['id'].isdigit() else (1, r['id'])

    rows = sorted(rows, key=sort_key)
    df = pd.DataFrame(rows)

    assert df.id.is_unique, 'TEST stems must be unique.'
    return df


@torch.no_grad()
def predict_snapshot(ckpt_path, test_loader):
    model = SourceRobustModel().to(DEVICE)

    state = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(state['model'], strict=True)
    model.eval()

    jenis_probs = []
    ker_probs = []

    for b in tqdm(
        test_loader,
        desc=f'Infer {Path(ckpt_path).name}',
        leave=False,
    ):
        x = b['pixel_values'].to(
            DEVICE,
            non_blocking=True,
        )

        # Backbone/model forward can stay in BF16/FP16.
        with torch.autocast(
            device_type='cuda',
            dtype=AMP_DTYPE,
        ):
            disaster_logits, _, cond_logits, _ = model(x)

        # IMPORTANT:
        # Recompute all probabilities in FP32 for numerical stability.
        disaster_prob = F.softmax(
            disaster_logits.float(),
            dim=-1,
        )

        cond_prob = F.softmax(
            cond_logits.float(),
            dim=-1,
        )  # [B, 3 disasters, 3 severities]

        # Soft routing in FP32.
        severity_prob = torch.sum(
            disaster_prob.unsqueeze(-1) * cond_prob,
            dim=1,
        )

        # Defensive normalization.
        disaster_prob = disaster_prob / disaster_prob.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-12)

        severity_prob = severity_prob / severity_prob.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-12)

        jenis_probs.append(
            disaster_prob.cpu().numpy()
        )
        ker_probs.append(
            severity_prob.cpu().numpy()
        )

    del model
    torch.cuda.empty_cache()

    jenis_probs = np.concatenate(jenis_probs).astype(np.float32)
    ker_probs = np.concatenate(ker_probs).astype(np.float32)

    assert np.isfinite(jenis_probs).all(), \
        f'Non-finite jenis probabilities in {ckpt_path}'

    assert np.isfinite(ker_probs).all(), \
        f'Non-finite kerusakan probabilities in {ckpt_path}'

    return jenis_probs, ker_probs


# ------------------------------------------------------------------
# TEST inference
# ------------------------------------------------------------------

test_df = enumerate_test(TEST_DIR)
print('TEST images:', len(test_df))

test_loader = DataLoader(
    TestDS(test_df),
    batch_size=EVAL_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)


# Snapshot epoch 3
j3, k3 = predict_snapshot(
    SNAP3,
    test_loader,
)

# Snapshot epoch 4
j4, k4 = predict_snapshot(
    SNAP4,
    test_loader,
)


# ------------------------------------------------------------------
# Probability-level snapshot ensemble
# ------------------------------------------------------------------

jenis_prob = 0.5 * j3 + 0.5 * j4
ker_prob = 0.5 * k3 + 0.5 * k4


# Normalize again after averaging, mostly defensive.
jenis_prob = jenis_prob / np.clip(
    jenis_prob.sum(axis=1, keepdims=True),
    1e-12,
    None,
)

ker_prob = ker_prob / np.clip(
    ker_prob.sum(axis=1, keepdims=True),
    1e-12,
    None,
)


# ------------------------------------------------------------------
# Validation
# ------------------------------------------------------------------

assert jenis_prob.shape == (len(test_df), 3)
assert ker_prob.shape == (len(test_df), 3)

assert np.isfinite(jenis_prob).all()
assert np.isfinite(ker_prob).all()

assert np.allclose(
    jenis_prob.sum(axis=1),
    1.0,
    atol=1e-6,
)

assert np.allclose(
    ker_prob.sum(axis=1),
    1.0,
    atol=1e-6,
)

print(
    'Jenis probability sum:',
    jenis_prob.sum(1).min(),
    '→',
    jenis_prob.sum(1).max(),
)

print(
    'Kerusakan probability sum:',
    ker_prob.sum(1).min(),
    '→',
    ker_prob.sum(1).max(),
)


# ------------------------------------------------------------------
# Save probabilities
# ------------------------------------------------------------------

np.savez_compressed(
    OUTPUT_DIR / 'test_probabilities_snapshot34.npz',

    ids=test_df.id.astype(str).to_numpy(),

    jenis_prob=jenis_prob,
    kerusakan_prob=ker_prob,

    jenis_prob_epoch3=j3,
    kerusakan_prob_epoch3=k3,

    jenis_prob_epoch4=j4,
    kerusakan_prob_epoch4=k4,
)

print('Saved probabilities.')

TEST images: 450


Infer partial_epoch_3.pt:   0%|          | 0/15 [00:00<?, ?it/s]

Infer partial_epoch_4.pt:   0%|          | 0/15 [00:00<?, ?it/s]

Jenis probability sum: 0.99999994 → 1.0000001
Kerusakan probability sum: 0.9999999 → 1.0000001
Saved probabilities.


## Build numeric submission

The sample-solution row order is preserved exactly. Output is a comma-separated `ID,Target` CSV.


In [12]:
pred_j=jenis_prob.argmax(1)
pred_k=ker_prob.argmax(1)
by_id={
    str(test_df.iloc[i].id):(
        SUB_JENIS[IDX_TO_JENIS[int(pred_j[i])]],
        SUB_KER[IDX_TO_KER[int(pred_k[i])]],
    )
    for i in range(len(test_df))
}

solution=pd.read_csv(SOLUTION_PATH,sep=None,engine='python')
assert solution.columns.tolist()==['ID','Target'], f'Unexpected solution columns: {solution.columns.tolist()}'
out=solution.copy()
target=[]
for rid in out.ID.astype(str):
    base,suffix=rid.rsplit('_',1)
    assert base in by_id, f'Missing TEST ID: {base}'
    if suffix=='jenis': target.append(by_id[base][0])
    elif suffix=='kerusakan': target.append(by_id[base][1])
    else: raise ValueError(f'Unexpected solution suffix: {suffix}')
out['Target']=target

SUBMISSION_PATH=OUTPUT_DIR/'submission.csv'
out.to_csv(SUBMISSION_PATH,index=False)
print(out.head(12).to_string(index=False))
print('\nshape:',out.shape)
print('Target counts:')
print(out.Target.value_counts().sort_index())
print('\nSaved:',SUBMISSION_PATH)


         ID  Target
    1_jenis       1
1_kerusakan       1
    2_jenis       1
2_kerusakan       1
    3_jenis       1
3_kerusakan       1
    4_jenis       1
4_kerusakan       1
    5_jenis       1
5_kerusakan       1
    6_jenis       1
6_kerusakan       2

shape: (900, 2)
Target counts:
Target
1    342
2    321
3    237
Name: count, dtype: int64

Saved: /workspace/output/siglip2_b384_source_robust_v2/submission.csv


## Final sanity checks


In [14]:
submission=pd.read_csv(SUBMISSION_PATH)
assert submission.columns.tolist()==['ID','Target']
assert len(submission)==len(solution)
assert submission.ID.astype(str).tolist()==solution.ID.astype(str).tolist()
assert submission.Target.notna().all()
assert set(submission.Target.astype(int).unique()).issubset({1,2,3})
assert len(test_df)*2==len(submission), 'Expected two output rows per TEST image.'

print('Submission ready:',SUBMISSION_PATH)
print('Snapshot 3:',SNAP3)
print('Snapshot 4:',SNAP4)
print('Probabilities:',OUTPUT_DIR/'test_probabilities_snapshot34.npz')


Submission ready: /workspace/output/siglip2_b384_source_robust_v2/submission.csv
Snapshot 3: /workspace/output/siglip2_b384_source_robust_v2/partial_epoch_3.pt
Snapshot 4: /workspace/output/siglip2_b384_source_robust_v2/partial_epoch_4.pt
Probabilities: /workspace/output/siglip2_b384_source_robust_v2/test_probabilities_snapshot34.npz
